# 🧭 生成式AI應用開發｜第04週｜150分鐘修訂版
## Prompt Engineering：從訊息結構到可驗證的小功能
延續第三周 Responses API，以固定 few-shot 範例銜接第五周對話紀錄。
本檔另存自原實作教材，原版保留。先備：list/dict、for、函式、instructions、role/content。

> **學生版**：訊息組裝、客服分類器與擇一應用保留 TODO；先完成練習再開啟付費區塊。

## 🎯 學習目標
1. 區分工作身分與訊息role，使用instructions搭配訊息列表。
2. 用五元素、分隔符與任務拆解寫出可測試的規格。
3. 完成引導練習：用迴圈組裝user/assistant範例，並離線檢查。
4. 完成練習A：客服分類器v1/v2，用同一批案例比較有無範例。
5. 完成練習B改寫器或練習C資料問答其中一題，檢查輸出與事實。
6. 分清API失敗、非法格式與答案錯誤；v3衝突範例為選做。

## 🗺️ 150分鐘課堂流程（有效授課時間，休息另計）
| 分鐘 | 區塊 | 完成要求 |
|---|---|---|
| 0–15 | 環境與第三周複習 | 規則、資料、訊息角色 |
| 15–35 | 基準比較、五元素 | 觀察與改良prompt |
| 35–55 | instructions、分隔符、任務拆解 | 讀懂並修改示範 |
| 55–80 | 短示範＋引導練習：訊息組裝 | 離線檢查9則訊息 |
| 80–110 | 練習A：客服分類器（必做） | v1單筆→v2單筆→五題比較 |
| 110–135 | 練習B改寫／練習C問答（擇一） | 所選題輸出與檢查 |
| 135–150 | 同儕覆核、修訂紀錄 | 說明結果與修改理由 |
摘要與課程意見分類只作示範；學生實作一個客服分類器，再完成B或C。
v3、未選題與額外挑戰均為延伸，不影響課堂完成判定。
新增離線小示範供理解語法，不需學生修改；詳細解說為選讀。仍以引導練習＋A＋B/C一題為課堂成果。


---

## 1️⃣ 環境準備

沿用第 3 週設定：API key 放在 Colab Secrets 的 `OPENAI_API_KEY`，不要直接寫進 Notebook。程式會先檢查金鑰，再建立 API client；若設定不完整，會提供明確錯誤訊息。


In [ ]:
# 安裝或更新 OpenAI Python SDK
!pip install -q --upgrade openai


In [ ]:
import os
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則使用作業系統環境變數。
try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境時，沿用作業系統環境變數。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")


# 沒有金鑰也能完成訊息列表與規格的離線練習；付費入口會明確提示。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI() if os.getenv("OPENAI_API_KEY") else None
print("本週模型：", MODEL)
print("API 已可用" if client else "離線練習模式：尚未設定 API key")

### 🧭 先看API程式的三層責任（共用程式，不是練習）
| 層次 | 函式 | 責任／回傳 |
|---|---|---|
| 輸入檢查 | require_text | 不合法就拋錯，沒有呼叫API |
| 共用請求 | request_text | 呼叫一次API，檢查完成／拒答／空文字，回傳文字 |
| 任務入口 | ask_ai／ask_ai_messages | 分別接受字串／訊息列表，再交給共用請求 |
摘要／改寫／問答→ask_ai→request_text；分類→ask_ai_messages→request_text。
每個入口正常執行只呼叫API一次，不是每層各呼叫一次。下列三格照順序執行定義，不需修改。

In [ ]:
def require_text(value, name):
    """在本機先拒絕空白或錯誤型別，避免無效的付費請求。"""
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} 必須是非空白字串。")

In [ ]:
def request_text(api_input, instructions):
    """共用 API 與錯誤處理；各入口先驗證自己的輸入結構。"""
    require_text(instructions, "instructions")
    if client is None:
        raise RuntimeError("請先設定 OPENAI_API_KEY 並重新執行環境 cell。")
    try:
        response = client.responses.create(
            model=MODEL, instructions=instructions, input=api_input, store=False,
        )
    except Exception as exc:
        # API 失敗與模型答錯分開處理，不把錯誤文字當成正常答案。
        raise RuntimeError(f"OpenAI API 呼叫失敗：{exc}") from exc
    if getattr(response, "status", "completed") != "completed":
        raise RuntimeError("回應尚未完成，不採用部分文字。")
    # 細節選讀對照：getattr處理缺屬性，or []處理None；下方離線fixture只示範這個讀法。
    for item in getattr(response, "output", []) or []:
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "refusal":
                raise RuntimeError("模型拒絕此請求，請檢查資料與任務。")
    result = response.output_text
    if not isinstance(result, str) or not result.strip():
        raise RuntimeError("模型沒有回傳可用文字。")
    return result

In [ ]:
def ask_ai(user_input, instructions="你是一位清楚、可靠的繁體中文助理"):
    """沿用單次文字任務入口，摘要、改寫與資料問答使用此函式。"""
    require_text(user_input, "user_input")
    return request_text(user_input, instructions)


def ask_ai_messages(messages, instructions):
    """接收有角色的訊息列表；本週只使用 user/assistant 固定範例。"""
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必須是非空列表。")
    for message in messages:
        if not isinstance(message, dict) or message.get("role") not in {"user", "assistant"}:
            raise ValueError("本週訊息 role 必須是 user 或 assistant。")
        require_text(message.get("content"), "content")
    if messages[-1]["role"] != "user":
        raise ValueError("最後一則必須是本次待回答的 user 訊息。")
    return request_text(messages, instructions)


def compare_prompts(user_input, prompt_variants):
    """同一筆文字搭配不同 instructions；每個版本各付費一次。"""
    if not isinstance(prompt_variants, dict) or not prompt_variants:
        raise ValueError("prompt_variants 必須是非空字典。")
    return {name: ask_ai(user_input, rules) for name, rules in prompt_variants.items()}

### 回應檢查怎麼讀？（細節選讀）

**對應哪一格？** 回到前面的 `def request_text(api_input, instructions):` code cell。本段解釋的是API回傳之後的檢查，不是 `ask_ai_messages` 的輸入檢查。摘要、分類、改寫與資料問答最後都會使用 `request_text`，因此共用這套檢查。

**先看原程式的三個步驟：**

| 原程式片段（依執行順序） | 讀法 | 為什麼需要？ |
|---|---|---|
| `getattr(response, "status", "completed") != "completed"` | 取得status，若不等於completed就拋錯 | 避免採用未完成的回答；缺少status時，這份helper以completed作為預設值，並非已確認回應完成 |
| `for item in getattr(response, "output", []) or []:` | 逐一讀取output中的項目 | 檢查回應是否含有拒答訊息 |
| `for part in getattr(item, "content", []) or []:` | 每個項目再逐一讀取content中的片段 | 拒答可能在片段中，不能只看最後的文字 |
| `getattr(part, "type", None) == "refusal"` | 若片段type是refusal就拋錯 | 拒答不當成正常任務答案 |
| `result = response.output_text` 與後面的if | 確認結果是非空str，最後return | 呼叫端取得可用文字，才進行分類或應用檢查 |

**把最容易看不懂的一行拆開：**

```python
# 對應原程式：for item in getattr(response, "output", []) or []:
raw_output = getattr(response, "output", [])
output_items = raw_output or []
for item in output_items:
    # 此處接原程式的content迴圈與拒答檢查。
    pass
```

`getattr`只在屬性不存在時使用預設值；屬性存在但值為None，取得的仍是None。接著`or []`把None等假值換成空列表，因此迴圈可跳過，不會因為遍歷None而出錯。content那一行使用相同讀法。這不是任意資料型別的驗證，例如非空整數仍不能用for遍歷。

**下一格離線示範如何對應原程式？**

| 下一格的名稱 | 原程式對應位置 |
|---|---|
| `fixture` | 代替response，只練習讀取output屬性 |
| `raw_value` | `getattr(response, "output", [])`的結果 |
| `safe_items` | 加上`or []`後，原程式實際迭代的內容 |

依序觀察三種虛構物件：缺少output時印出`[] → []`；output為None時印出`None → []`；output為列表時保留原列表。再回到 `request_text`，找出output及content兩處相同寫法。

下格不連線，只示範屬性讀取與None處理，沒有模擬完整回應、status或拒答。API呼叫本身失敗由前面的try/except處理；回應未完成、拒答或空文字則由本段檢查處理。不需改動共用helper。


In [ ]:
# 細節選讀：對照request_text讀取response.output的寫法；本格使用虛構物件，不呼叫API。
from types import SimpleNamespace
# 少了output與output為None是不同情況；兩者經過or []後可安全迭代。
# 三種情況依序是：沒有output屬性、output為None、output已有列表。
attribute_fixtures = [SimpleNamespace(), SimpleNamespace(output=None), SimpleNamespace(output=["示範項目"])]
for fixture in attribute_fixtures:
    # 對照getattr(response, "output", [])：缺屬性用[]，有屬性但為None仍取得None。
    raw_value = getattr(fixture, "output", [])
    # 對照正式迴圈中的or []：把None換成可迭代空列表，已存在的非空列表則保留。
    safe_items = raw_value or []
    print("原值：", raw_value, "｜迴圈使用：", safe_items)

> **執行原則**：依每段旁邊的步驟操作，只開啟當前活動的開關，不必重跑整份Notebook。
> 定義函式不付費；明列的API示範／測試才付費。完成依賴的TODO再開啟該題。
> 各段開關預設False，成本以請求數與實際usage判讀；store=False仍會傳送資料。
> 一次分類相同不代表範例無效；加入few-shot也不保證分數提高。

---

## 2️⃣ 從模糊 prompt 建立基準結果

先保留簡單版本作為 baseline。之後每次只調整一個主要變因，才能判斷是哪項改動造成差異。


In [ ]:
course_intro = """
生成式AI應用開發是一門大三課程，學生會使用 Python、OpenAI API、
Streamlit 與 GitHub 完成 AI 應用。課程重視實作、API key 安全、成本控制，
並以期中個人專題和期末小組專題整合所學。
"""

# 用 XML-like 標籤包住資料，可降低模型把指令與資料混在一起的機率。
comparison_input = f"""請摘要以下課程資料：
<course>
{course_intro}
</course>"""
# baseline prompt 故意保持簡短，稍後用來和明確版本比較差異。
baseline_instructions = "請摘要使用者提供的課程資料。"
# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_BASELINE = False
if RUN_BASELINE:
    baseline_result = ask_ai(comparison_input, baseline_instructions)
    print(baseline_result)

### ✍️ 觀察紀錄

- 摘要有幾句？是否使用繁體中文？
- 是否保留課程工具與評量方式？
- 重跑一次時，長度與格式是否相同？

模糊 prompt 不一定失敗，但它沒有先定義「什麼叫成功」。


---

## 3️⃣ Prompt 的五個實用元素

| 元素 | 要回答的問題 | 範例 |
|---|---|---|
| 角色 Role | 模型以什麼身份工作？ | 大學課程助教 |
| 任務 Task | 具體完成什麼？ | 摘要課程介紹 |
| 背景 Context | 必要資料是什麼？ | `<course>...</course>` |
| 限制 Constraints | 長度、語言、禁忌？ | 60 字內、不新增資訊 |
| 輸出 Output | 結果長什麼樣？ | 一句總結＋三個重點 |

口訣：**誰、做什麼、根據什麼、遵守什麼、輸出什麼。**


In [ ]:
# 明確版本把角色、資料限制、輸出長度與重點數都寫進 instructions。
clear_instructions = """
# Identity
你是大學課程的教學助理，擅長整理繁體中文課程資訊。

# Instructions
1. 只能根據使用者提供的課程文字摘要，不得新增資訊。
2. 先寫一句 60 字以內的總結。
3. 再列三個重點，涵蓋技術、課程特色與評量。
4. 使用繁體中文。
"""
# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_CLEAR_PROMPT = False
if RUN_CLEAR_PROMPT:
    clear_result = ask_ai(comparison_input, clear_instructions)
    print(clear_result)
# 已保留兩份輸出，不另重複呼叫 compare_prompts；它可供課後新版本比較。

### ✍️ 預測、觀察、修正（兩版皆填）
執行前先圈出五元素；預測明確版本可能改善哪一項，再比較實際結果。
| 面向 | 基準實際結果 | 明確版實際結果 | 通過條件 |
|---|---|---|---|
| 格式 |  |  | 一句總結＋三點 |
| 長度 |  |  | 總結 60 字元內 |
| 必要資訊 |  |  | 技術、特色、評量 |
| 事實 |  |  | 沒有新增原文未提供的資訊 |
不預填「一定改善」；結果可能相同或退步。下一版只調整一個主要變因。
字數以 Python 字元數計，包含標點；換行可排除，但不等於 token。

---

## 4️⃣ `instructions` 與 `input`：規則和資料分開

- `instructions`：應用程式提供的固定角色、規則、流程與範例。
- `input`：每次請求會改變的使用者任務與資料。

課堂常把固定高層規則泛稱為 system prompt；本課使用 Responses API，因此用 `instructions` 實作。可把固定規則想成函式定義，把每次輸入想成函式參數。


### 🔗 接回第三周：工作身分與訊息 role
| 名稱 | 意義 | 本週放法 |
|---|---|---|
| 工作身分 | 助教、摘要器、分類器 | instructions 的角色描述 |
| developer | 開發者提供的高層規則 | 第三周示範；本週規則統一放 instructions |
| user | 範例輸入或本次任務 | input 列表中的訊息 |
| assistant | 提供給模型的範例答案／先前回答 | 本週用人工提供的示範答案 |
`input` 可以是字串，也可以是訊息列表。`instructions` 和訊息列表可以一起使用。
本週的 assistant 範例是事先寫好的，沒有在此新增一次 API 呼叫，也不是訓練模型。
第五周才累積真正的 user/assistant 歷史；每次只送相同固定範例不會自動記住上一輪。

In [ ]:
# 先印出請求結構；只開啟 RUN_ROLE_DEMO 才真的送出一次請求。
bridge_messages = [{"role": "user", "content": "請用一句話解釋 API。"}]
print(bridge_messages)
# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_ROLE_DEMO = False
if RUN_ROLE_DEMO:
    print(ask_ai_messages(bridge_messages, "你是資管系助教，使用繁體中文與生活化例子。"))

### 摘要函式的資料流（示範，不需補TODO）
| 輸入／變數 | 放在哪裡 | 用途 |
|---|---|---|
| text | input的source_text | 這次要摘要的原文 |
| num_points | instructions的f-string | 重點數規格 |
| max_chars_per_point | instructions的f-string | 每點字元上限 |
| ask_ai回傳文字 | summarize_text的return | 交給呼叫端顯示／檢查 |
原教材也是「參數→規則與資料→API→回傳文字」。f-string會立即代入變數；XML標籤只是文字分隔符。
`return`把值交回呼叫端，`print`只顯示；後續練習需保存結果，所以不能只print而忘記return。
字元數以Python len計，不等於token；prompt提出限制，程式／人工仍要檢查。

In [ ]:
def summarize_text(text, max_chars_per_point=35, num_points=3):
    # 將可調整的輸出限制設成參數，讓同一個函式能重複用在不同資料。
    if max_chars_per_point <= 0 or num_points <= 0:
        raise ValueError("字數與重點數必須是正整數。")

    instructions = f"""
# Identity
你是精準的繁體中文摘要助理。
# Instructions
- 只根據提供的文字作答，不新增內容。
- 保留重要名稱、數字與限制。
- 輸出一個標題與 {num_points} 個條列重點。
- 每點不超過 {max_chars_per_point} 字。
"""
    # 原文放在 input；摘要規則放在 instructions，維持 prompt 結構清楚。
    user_input = f"""請摘要：
<source_text>
{text}
</source_text>"""
    return ask_ai(user_input, instructions)



# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_SUMMARY_DEMO = False
if RUN_SUMMARY_DEMO:
    print(summarize_text(course_intro, max_chars_per_point=35, num_points=3))

### 🧱 Markdown 與 XML 分隔資料

- Markdown 標題讓 prompt 層次清楚、容易維護。
- XML 標籤指出資料與範例的開始、結束。
- 分隔符能降低規則與資料混淆，但不是完整安全機制；外部文字仍是不可信資料。


---

## 5️⃣ 任務拆解：把複雜要求排成步驟

當任務包含多個判斷，明確寫出可觀察的工作流程，通常比只寫「仔細思考」更容易測試。


In [ ]:
feedback = "介面很漂亮，但註冊流程一直失敗，客服兩天後才回覆。"
# Workflow 先要求模型按步驟思考，Output format 再約束最後輸出欄位。
analysis_instructions = """
# Identity
你是產品回饋分析助理。
# Workflow
1. 找出稱讚項目。2. 找出問題。3. 判斷最優先問題。4. 提出一項改善建議。
# Output format
- 優點：
- 問題：
- 優先處理：
- 建議：
"""

# 本段示範：1次API請求；不依賴後面的學生練習。
RUN_WORKFLOW_DEMO = False
if RUN_WORKFLOW_DEMO:
    print(ask_ai(f"<feedback>{feedback}</feedback>", analysis_instructions))

### 小任務：先讀規格再執行
請把任務拆解的四步對應到四個輸出欄位；自行指出一項原文無法支持的推測。
分隔符提升可讀性，不能保證防止提示注入；資料中的指令仍應視為資料。

---

## 6️⃣ Zero-shot 與 Few-shot

- **Zero-shot**：描述規則，不提供答案範例。
- **Few-shot**：提供少量且具代表性的輸入／輸出範例，讓模型掌握標籤、格式與邊界。

範例應涵蓋多樣情況；太相似的範例無法清楚呈現分類界線。


### Few-shot 的兩種呈現方式
原版以 `<example><input>…</input><output>…</output></example>` 寫進高層指令，這也是有效方式。
本版主要用 `user`／`assistant` 訊息配對，練習第三周的 list/dict 與第五周會用到的訊息結構。
兩種方式沒有保證孰優；本次實驗固定用訊息列表，只改有無範例，避免同時改兩個變因。

### 教師短示範：同一規則，三筆未見過的留言
**目標**：觀察範例是否釐清分類邊界，不要求兩版一定回答不同。
**操作**：先閱讀規則並預測三題答案；下格程式完整提供，不需修改。
開啟RUN_FEWSHOT_DEMO後執行，共3題×2版本＝6次請求。
一般題容易同樣答對；邊界題比較「增加教材說明」與「增加練習時間」，不要把多類衝突當唯一例子。
本示範不依賴尚未完成的build_few_shot_messages；學生只在下一段練習迴圈組裝。
`COURSE_CATEGORIES`是合法類別清單：用來產生規則中的類別文字，也用來檢查模型回覆。
「合法標籤」只檢查回覆是否為四類之一；「符合預期」才比較分類是否正確。


In [ ]:
COURSE_CATEGORIES = ["內容", "進度", "設備", "其他"]
# 從同一份類別清單產生規則，並在下方檢查回覆，避免兩處定義不一致。
COURSE_RULES = f"""你是課程意見分類器，只輸出{"、".join(COURSE_CATEGORIES)}其中一類。
教材內容、概念說明與例子歸內容；操作節奏與練習時間歸進度；設備或網路歸設備；其餘歸其他。
"""
# 代表性範例用明確訊息配對示範；後面的測試題沒有出現在這裡。
COURSE_EXAMPLES = [
    ("同一個概念希望用不同案例再解釋。", "內容"),
    ("觀念已懂，但還沒操作完就進下一段。", "進度"),
    ("教室網路連不上。", "設備"),
    ("希望下次自由分組。", "其他"),
]
COURSE_EXAMPLE_MESSAGES = [
    {"role": "user", "content": "同一個概念希望用不同案例再解釋。"},
    {"role": "assistant", "content": "內容"},
    {"role": "user", "content": "觀念已懂，但還沒操作完就進下一段。"},
    {"role": "assistant", "content": "進度"},
    {"role": "user", "content": "教室網路連不上。"},
    {"role": "assistant", "content": "設備"},
    {"role": "user", "content": "希望下次自由分組。"},
    {"role": "assistant", "content": "其他"},
]
COURSE_DEMO_TESTS = [
    {"input": "投影機沒有畫面。", "expected": "設備"},
    {"input": "希望多留十分鐘讓我們自己試一次。", "expected": "進度"},
    {"input": "API觀念仍不清楚，希望用完整程式再解釋一次。", "expected": "內容"},
]
RUN_FEWSHOT_DEMO = False
if RUN_FEWSHOT_DEMO:
    for case in COURSE_DEMO_TESTS:
        current_user = {"role": "user", "content": case["input"]}
        # 固定規則與測試題，只改有無示範訊息。
        zero = ask_ai_messages([current_user], COURSE_RULES).strip()
        few = ask_ai_messages(COURSE_EXAMPLE_MESSAGES + [current_user], COURSE_RULES).strip()
        print("題目：", case["input"], "｜預期：", case["expected"])
        # 合法標籤仍可能分類錯誤，因此分開顯示合法性與是否符合預期。
        for method, result in [("zero-shot", zero), ("few-shot", few)]:
            print(method, "：", result,
                  "｜合法標籤：", result in COURSE_CATEGORIES,
                  "｜符合預期：", result == case["expected"])

### ✍️ 短示範觀察表
| 題型 | 預期 | zero-shot實際 | few-shot實際 | 有無改善／原因 |
|---|---|---|---|---|
| 設備一般題 | 設備 |  |  |  |
| 練習時間邊界題 | 進度 |  |  |  |
| 概念說明邊界題 | 內容 |  |  |  |
若兩版相同，記錄「本次沒有改善」，並指出範例展示了哪個分類邊界。
這三題只是示範，不足以證明few-shot的整體效果；不要刻意弱化zero-shot來製造差異。

### 引導練習：訊息組裝（必做、不付費）
**情境**：把固定範例轉成API訊息列表。不是呼叫模型，也不產生分類答案。
| 名稱 | 型別／內容 | 誰提供 |
|---|---|---|
| text | str，本次待分類的留言 | 呼叫端 |
| examples | list，每筆是(範例輸入, 標準標籤) tuple | 已提供COURSE_EXAMPLES |
| 回傳值 | 新list，每則dict含role與content | 學生完成函式 |
**要完成的項目**：G1建立空列表→G2逐筆拆開範例並加入user/assistant→G3加入本次user並回傳。
四組範例應回傳9則；examples=[]應只回傳本次user共1則，不能把本次答案加進列表。
**執行**：看下方小示範→完成G1/G2/G3→執行函式定義→RUN_MESSAGE_CHECK改True並執行。
**完成判定**：角色交錯、最後user內容等於text、不修改examples；通過後A直接重用此函式。

In [ ]:
# 離線語法示範：不同情境的一組範例，不提供整個練習函式的答案。
toy_pair = ("這杯茶很好喝。", "正向")
toy_input, toy_label = toy_pair  # tuple解包，左邊兩個變數各接一個值。
toy_messages = []
# 對照訊息組裝練習G2：extend逐一加入兩則dict，維持user／assistant配對。
toy_messages.extend([
    {"role": "user", "content": toy_input},
    {"role": "assistant", "content": toy_label},
])
# 對照G3：本次問題只有一則user，用append加入；不能加入尚未取得的答案。
toy_messages.append({"role": "user", "content": "味道有點淡。"})
print("一組範例＋本次問題：", toy_messages)
# append加入一個項目；extend把列表中的項目逐個加入；+建立合併後的新列表。
# [toy_messages]示意把整個列表當成單一項目加入的巢狀結構；這行本身沒有呼叫append。
print("append列表會變巢狀：", [toy_messages])
# +建立新的外層列表，原toy_messages不增加項目；內部dict仍共用，並非深層複製。
print("用+接新訊息：", toy_messages + [{"role": "user", "content": "另一則問題"}])

In [ ]:
def build_few_shot_messages(text, examples):
    """練習：將範例配對轉成訊息列表；此步驟不呼叫 API。"""
    require_text(text, "text")
    # TODO G1：建立空 messages 列表。
    # TODO G2：用 for 解開每個 (sample_input, sample_output)，附上 user/assistant 配對。
    # TODO G3：最後附上本次 text 的 user 訊息，回傳 messages。
    # 提示：四組範例會產生 8 則示範訊息，再加 1 則本次輸入。
    raise NotImplementedError("請完成訊息組裝練習，再開啟相關示範。")

In [ ]:
RUN_MESSAGE_CHECK = False
if RUN_MESSAGE_CHECK:
    # 此檢查不呼叫 API，完成組裝函式後可先開啟。
    messages = build_few_shot_messages("練習時間不夠。", COURSE_EXAMPLES)
    assert len(messages) == 9
    assert [m["role"] for m in messages] == ["user", "assistant"] * 4 + ["user"]
    assert messages[-1]["content"] == "練習時間不夠。"
    print(messages)

### 引導練習完成紀錄
- [ ] 9則訊息與角色順序檢查通過。
- [ ] 能指出人工範例答案與本次問題的位置。
若出現NotImplementedError，先完成函式，再進練習A；本段沒有API成本。

<div style="border-left:5px solid #f9ab00;padding:10px 14px;background:#fff8e1;">
<font color="#b06000"><b>先不要混淆</b></font>：本週只用文字規則限制輸出，仍可能出現意外格式；第 6 週再用 Structured Outputs 與 schema 做程式層級保證。
</div>


## 🛠️ 練習A：客服分類器（必做，80–110分）
**情境**：把客服留言分到四類；做同一個分類器的v1無範例與v2有範例比較。
**已提供**：SUPPORT_LABELS、五筆SUPPORT_TESTS、API helper、版本包裝與固定測試器，不需重寫。
**學生負責**：規則字串、classify_support_message與每類範例；不修改測試題來追求通過。
| 步驟 | TODO／操作 | 具體成果 |
|---|---|---|
| A1 | SUPPORT_RULES_V1 | 身分、四類定義、衝突優先序、只輸出標籤 |
| A2a | 組裝與呼叫 | message＋examples→訊息列表→模型文字→strip |
| A2b | 檢查與回傳 | 合法標籤回傳str；不合法raise ValueError |
| A3 | 開RUN_SUPPORT_V1_SINGLE | v1單筆能執行，1次請求 |
| A4 | SUPPORT_EXAMPLES_V2＋單筆 | 每類至少一例，不抄測試題，1次請求 |
| A5 | 開RUN_SUPPORT_COMPARISON | 同五題比較v1/v2，10次請求 |
| A6 | 比較表 | 說明相同／改善／退步與錯誤類型 |
衝突優先序固定為帳務問題→功能問題→操作疑問→其他；v3為選做。

### A的函式規格與完成範例（先讀再填）
`classify_support_message(message, examples)`：message是留言str，examples是(input, label)配對list。
固定規則使用SUPPORT_RULES_V1；本次資料由組裝函式放入訊息列表，不把留言接進高層規則。
回傳必須是四類之一的str，不是response物件、訊息列表或print的None。
例如「付款金額與帳單不符」依規格應回傳「帳務問題」；不要求學生先寫死這個答案。
v1傳examples=[]，v2傳四類範例；同一個分類函式及規則保持不變，只改有無範例。
每個範例tuple第一項是留言、第二項是合法標籤；標籤須精確拼寫，不能加說明文字。

In [ ]:
SUPPORT_LABELS = ["功能問題", "帳務問題", "操作疑問", "其他"]
# 範例與測試集分開：不要把後面的測試題直接放進範例。
SUPPORT_TESTS = [
    {"input": "我的信用卡被扣款兩次。", "expected": "帳務問題"},
    {"input": "我要在哪裡修改登入密碼？", "expected": "操作疑問"},
    {"input": "按儲存後出現錯誤，資料消失了。", "expected": "功能問題"},
    {"input": "謝謝客服快速回覆！", "expected": "其他"},
    {"input": "扣款兩次，而且報表頁面打不開。", "expected": "帳務問題"},
]

# TODO A1：寫出分類器身分、四類定義、衝突優先序與只輸出標籤的限制。
SUPPORT_RULES_V1 = ""

In [ ]:
def classify_support_message(message, examples):
    # TODO A2a：組裝messages，再用SUPPORT_RULES_V1呼叫ask_ai_messages，strip回覆。
    # TODO A2b：不在SUPPORT_LABELS就raise ValueError；合法時return分類文字。
    raise NotImplementedError("請完成練習A分類函式。")


def support_v1(text):
    return classify_support_message(text, [])

In [ ]:
# A3：完成規則與分類函式後開啟；只測一題，1次API請求。
RUN_SUPPORT_V1_SINGLE = False
if RUN_SUPPORT_V1_SINGLE:
    print(support_v1(SUPPORT_TESTS[0]["input"]))

### A4範例檢查的讀法（語法選讀）

**對應哪一格？** 本段與下一格離線示範，都是為了讀懂其後包含 `SUPPORT_EXAMPLES_V2`、`support_v2` 和 `RUN_SUPPORT_V2_SINGLE` 的code cell。正式檢查在呼叫 `support_v2` **之前**：

```python
if len(SUPPORT_EXAMPLES_V2) < 4 or {label for _, label in SUPPORT_EXAMPLES_V2} != set(SUPPORT_LABELS):
    raise ValueError("請先完成每類至少一例，並使用合法標籤。")
```

| 原程式片段 | 檢查內容 |
|---|---|
| `len(SUPPORT_EXAMPLES_V2) < 4` | 少於四筆範例就不通過 |
| `for _, label in SUPPORT_EXAMPLES_V2` | 每筆(input, label)解包；這裡只使用label，`_`仍是普通變數名稱 |
| `{label for _, label in SUPPORT_EXAMPLES_V2}` | 收集出現過的標籤，用集合去除重複 |
| `set(SUPPORT_LABELS)` | 將正式四類清單轉成集合，作為比較依據 |
| 兩個集合`!=` | 少了某類或出現非法標籤，都不通過；集合比較不看順序 |
| 兩項條件之間的`or` | 筆數不足或標籤不完整，任一成立就拋錯，不呼叫模型 |

**下一格離線示範如何對應原程式？**

| 離線示範 | 正式練習 |
|---|---|
| `toy_examples`的飲料意見配對 | `SUPPORT_EXAMPLES_V2`的客服留言配對 |
| 一般for加上`toy_seen_labels.add(toy_label)` | 把集合生成式展開，方便逐步閱讀 |
| `toy_compact_labels` | 正式條件中的`{label for _, label in SUPPORT_EXAMPLES_V2}` |
| `set(["正向", "負向"])` | 正式條件中的`set(SUPPORT_LABELS)` |

執行小示範後，一般for與集合生成式都得到正向、負向兩類，即使正向出現兩次也只保留一次；列印順序可能不同。示範只比較兩類，因此不能直接拿它的資料或標籤填入四類客服練習。

回到正式A4：完成每類至少一筆(input, label)，再開啟 `RUN_SUPPORT_V2_SINGLE`。例如四筆都標成「帳務問題」，筆數足夠但集合只有一類，仍不通過；四類都有但多了一筆非法標籤，也不通過。

檢查通過只代表筆數與標籤集合符合規格，不代表範例分類正確，也不驗證每筆配對的完整型別。留言與標籤是否合理仍需人工閱讀；後續A5才用固定題目觀察模型表現。學生只需完成範例，不需修改這個檢查條件。


In [ ]:
# 語法選讀：對照A4範例檢查；先展開一般for，再比較集合生成式，不呼叫API。
toy_examples = [("很好喝", "正向"), ("味道不好", "負向"), ("很喜歡", "正向")]
toy_seen_labels = set()
# 每筆tuple解包成留言與標籤，集合只收標籤；相同標籤重複加入不增加種類。
for toy_text, toy_label in toy_examples:
    toy_seen_labels.add(toy_label)
# 與上面的for等價；_接到留言但不使用，對照正式的{label for _, label in SUPPORT_EXAMPLES_V2}。
toy_compact_labels = {label for _, label in toy_examples}
print("一般for：", toy_seen_labels, "｜集合生成式：", toy_compact_labels)
# 集合不看順序；兩種寫法都應得到兩類。客服正式練習仍須涵蓋四類，不能照抄此資料。
assert toy_seen_labels == toy_compact_labels == {"正向", "負向"}
# 三筆範例只有兩類；數量夠不代表每個預期類別都已涵蓋。
print("有正向／負向：", toy_seen_labels == set(["正向", "負向"]))

In [ ]:
# TODO A4：每類各寫一筆(input, label)；不要複製SUPPORT_TESTS。
SUPPORT_EXAMPLES_V2 = []


def support_v2(text):
    return classify_support_message(text, SUPPORT_EXAMPLES_V2)

# A4：完成每類一例後，先做單筆檢查；1次API請求。
RUN_SUPPORT_V2_SINGLE = False
if RUN_SUPPORT_V2_SINGLE:
    if len(SUPPORT_EXAMPLES_V2) < 4 or {label for _, label in SUPPORT_EXAMPLES_V2} != set(SUPPORT_LABELS):
        raise ValueError("請先完成每類至少一例，並使用合法標籤。")
    print(support_v2(SUPPORT_TESTS[0]["input"]))

### A5：固定測試（已提供完整程式，不需補TODO）
對每一題：讀input→呼叫分類函式→取得actual→與expected比較→保存／顯示結果。
expected是依規格指定的標準標籤；actual是模型經過合法標籤檢查後的回覆。
合法但不符合expected是答案錯誤；非法標籤是契約錯誤；API失敗則另列，不當成prompt退步。
下方先用不付費的玩具函式理解「把函式交給別人呼叫」，再讀正式測試器。
完成A3/A4單筆後，開RUN_SUPPORT_COMPARISON，共10次請求；不修改教師提供的測試器。

### `for version, classify_fn in {"v1": support_v1, "v2": support_v2}.items():`（語法選讀）

**對應位置**：下方正式code cell的 `if RUN_SUPPORT_COMPARISON:` 區塊。這行讓同一個測試器依序測試v1與v2；不是另外建立兩個模型，也不是在字典建立時呼叫API。

| 程式片段 | 代表什麼？ |
|---|---|
| `{"v1": support_v1, "v2": support_v2}` | 字典：key是版本名稱str，value是已定義的分類函式物件 |
| `support_v1`／`support_v2`沒有括號 | 保存函式本身，之後才執行；若寫成`support_v1("留言")`，就會立刻呼叫並取得分類文字 |
| `.items()` | 回傳字典項目檢視物件；迭代時逐筆提供(key, value)配對，不是回傳list |
| `for version, classify_fn in ...` | 每次把配對解包到兩個變數：version接名稱、classify_fn接函式 |

**每次迴圈變數的值：**

| 次數 | version | classify_fn | 後續測試 |
|---|---|---|---|
| 第一次 | `"v1"` | support_v1函式 | 同五題，無範例 |
| 第二次 | `"v2"` | support_v2函式 | 同五題，有四類範例 |

`classify_fn`只是變數名稱，不是新的函式定義。第一輪指向support_v1，第二輪指向support_v2。可先把原行拆成下列讀法（僅供理解，不需另加執行格）：

```python
versions = {"v1": support_v1, "v2": support_v2}
for version, classify_fn in versions.items():
    version_results[version] = run_classification_tests(
        SUPPORT_TESTS, classify_fn, version
    )
```

**函式何時真正執行？** 回到同一格上方的測試器：

1. 外層迴圈將classify_fn交給 `run_classification_tests`，此時是傳入函式物件。
2. 測試器內層 `for case in test_cases:` 逐題取出固定測試資料。
3. 到 `actual = classify_fn(case["input"])` 才執行分類函式：v1走無範例路徑，v2走有範例路徑。
4. 測試器回傳該版本的結果列表，分別存入 `version_results["v1"]` 與 `version_results["v2"]`。

外層2個版本 × 內層5題，正常跑完共10次API請求；不是外層只有兩次就只呼叫兩次API。版本名稱用於紀錄，真正決定分類方式的是classify_fn指向的函式。

**與下一格離線小示範的關聯：**

| 小示範 | 正式測試 |
|---|---|
| toy_operations | v1／v2的版本字典 |
| name、operation | version、classify_fn |
| toy_apply接收operation | run_classification_tests接收classify_fn |
| `operation(text)` | `classify_fn(case["input"])` |

小示範只計算字元數，不呼叫API。先執行它理解「傳入函式→稍後呼叫」，再回到正式code cell找出外層選版本、內層逐題呼叫的位置；完成A3/A4後才開RUN_SUPPORT_COMPARISON，不需修改測試器。


In [ ]:
# 語法選讀：對照A5版本字典與固定測試；本格只算字元數，不呼叫API。
def toy_count_chars(text):
    return len(text)


def toy_apply(text, operation):
    # operation接收到函式物件，直到這一行才執行它。
    return operation(text)


# value放函式物件而非呼叫結果，對照{"v1": support_v1, "v2": support_v2}。
toy_operations = {"字元數": toy_count_chars}
# items逐筆提供(key, value)；解包為name與operation，對照version與classify_fn。
for name, operation in toy_operations.items():
    # 把operation交給toy_apply，直到它內部operation(text)才執行；正式測試器也如此逐題呼叫。
    print(name, toy_apply("API", operation))  # 預期印出字元數3。
# dict.items()回傳項目檢視物件；每次迭代可解包成名稱與函式。
# toy_count_chars是函式；toy_count_chars("API")是立即執行後取得的整數。

In [ ]:
def run_classification_tests(test_cases, classify_fn, version):
    """接受分類函式，讓同一批案例能測不同版本；保留錯誤類別便於判讀。"""
    rows = []
    for case in test_cases:
        try:
            # 這裡才真正呼叫v1或v2；外層選版本、內層逐題，共2個版本×5題。
            actual = classify_fn(case["input"])
            passed = actual == case["expected"]
            status = "通過" if passed else "模型答案錯誤"
        except NotImplementedError:
            # 尚未完成練習時明確中止，不把空白答案當作分類失敗。
            raise
        except RuntimeError as exc:
            actual, passed, status = str(exc), False, "API／回應失敗"
        except ValueError as exc:
            actual, passed, status = str(exc), False, "輸入／輸出契約錯誤"
        row = dict(version=version, input=case["input"], expected=case["expected"],
                   actual=actual, passed=passed, status=status, model=MODEL)
        rows.append(row)
        print(row)
    print(f"{version}：{sum(row['passed'] for row in rows)}/{len(rows)}")
    return rows



RUN_SUPPORT_COMPARISON = False
version_results = {}
if RUN_SUPPORT_COMPARISON:
    # 必做比較只有v1/v2；先檢查範例完整，避免空範例被誤當成few-shot。
    if len(SUPPORT_EXAMPLES_V2) < 4 or {label for _, label in SUPPORT_EXAMPLES_V2} != set(SUPPORT_LABELS):
        raise ValueError("請先完成每類至少一例，再做版本比較。")
    # 語法選讀對照：字典value是函式，items配對解包後，將該函式交給同一測試器。
    for version, classify_fn in {"v1": support_v1, "v2": support_v2}.items():
        version_results[version] = run_classification_tests(SUPPORT_TESTS, classify_fn, version)

### A6：比較與修訂紀錄
| 版本 | 改動 | 通過 | API失敗 | 契約錯誤 | 保留／退回與原因 |
|---|---|---|---|---|---|
| v1 | 明確規則，無範例 | /5 |  |  |  |
| v2 | 相同規則＋四類範例 | /5 |  |  |  |
即使兩版同分也可完成本題；評量依據是能否說明規則、範例與測試結果。
API失敗不是prompt退步；一般合法標籤但分類錯誤，才是答案判斷問題。
- [ ] 已完成規則與合法標籤檢查。
- [ ] 已完成每類一例與v1/v2比較。
- [ ] 已記錄一個觀察或待驗證條件。

## 🔀 練習B／C擇一（110–135分）
練習A完成後，選B改寫器或C資料問答；直接到所選區塊。
每個區塊依「示範→TODO→執行→檢查→紀錄」排列；未選題跳過，課後選讀。
示範與練習分開開關，只開啟當前題，不必重跑另一題。

## ✍️ 練習B：學習內容改寫器（與C擇一）
**情境**：把technical_text改寫給初次接觸主題的大學生。原文與輸出檢查器已提供。
**成果**：完成模板與rewrite_for_beginner；函式回傳完整str，呼叫端保存beginner_result。
| 步驟 | TODO | 實際要完成 |
|---|---|---|
| B1 | 模板文字 | 身分、白話解釋、保留名詞、一個比喻、理解題不給答案、{max_chars}及兩個標題 |
| B2 | 函式驗證 | max_chars不大於0就raise ValueError |
| B3 | 規則／資料／呼叫 | format代入限制，technical_text放input，呼叫ask_ai取得result |
| B4 | 檢查與回傳 | 排除換行計長度，超限印警告，return result |
| B5 | 執行區 | 呼叫函式並保存beginner_result，再print |
**執行**：讀下格通知改寫示範（可不付費）→完成TODO→執行定義→開RUN_PRACTICE_B→下一格檢查。
示範1次、練習1次API請求。完成B觀察表後跳同儕覆核，不需執行C。

In [ ]:
def rewrite_notice(text, audience="大學生", tone="友善", max_chars=100):
    # audience、tone、max_chars 參數化後，同一個改寫函式可套用不同情境。
    if max_chars <= 0:
        raise ValueError("max_chars 必須是正整數。")

    instructions = f"""
# Identity
你是校園行政文字編輯。
# Instructions
- 改寫成適合{audience}閱讀的{tone}繁體中文通知。
- 不得更改日期、時間、地點與應繳項目。
- 輸出一個短標題及一段通知，總字數不超過 {max_chars} 字。
"""
    result = ask_ai(f"<original_notice>{text}</original_notice>", instructions)
    # 模型可能超出字數限制，因此仍要在程式端做第二層檢查。
    if len(result.replace("\n", "")) > max_chars:
        print(f"⚠️ 字數檢查：模型輸出超過 {max_chars} 字，請調整 prompt 或改用結構化流程。")
    return result


notice = "本課程作業二應於10月15日23:59前上傳Moodle，逾期系統關閉，不收email補交。"

RUN_REWRITE_DEMO = False
if RUN_REWRITE_DEMO:
    print(rewrite_notice(notice, audience="大學生", tone="友善", max_chars=100))

### B模板、結果與成功條件
模板是含占位符的str，不是Python答案；`.format(max_chars=250)`把`{max_chars}`換成250。
例如`"最多{limit}字".format(limit=20)`得到「最多20字」；f-string則在建立字串時代入目前變數。
原文text另外放在input，不能只把字數代入模板卻忘了傳原文。
輸出應有「## 白話說明」及「## 想一想」，保留Embedding、解釋與原文一致，最後有一題理解題而不附答案。
B4字元數超限只印警告，不自動修好輸出；需修改prompt再觀察。檢查通過也不代表事實正確。
B5須指定beginner_result；只print呼叫結果會讓後面的檢查找不到要檢查的文字。

In [ ]:
# 練習 B：用 template 保留 prompt 結構，只把 max_chars 當成變數代入。
BEGINNER_REWRITE_PROMPT_TEMPLATE = """
# TODO B1：加入 Identity、Instructions、{max_chars} 參數與 Output format
"""


def rewrite_for_beginner(text, max_chars=250):
    # 提示：先檢查 max_chars，再用 template.format(max_chars=max_chars) 產生 instructions。
    # TODO B2：檢查 max_chars 必須大於 0
    # TODO B3：用 format() 代入字數，用 <technical_text> 包住資料並呼叫 ask_ai
    # TODO B4：若輸出超過 max_chars，印出警告；最後 return result
    # 提示：prompt 限制不是保證，程式仍需做字數檢查。
    raise NotImplementedError("請完成改寫器。")


technical_text = """
Embedding 是把文字轉換成高維度數值向量的技術。語意相近的文字，
其向量在空間中的距離通常較近，因此可以用來進行語意搜尋。
"""

RUN_PRACTICE_B = False
if RUN_PRACTICE_B:
    # TODO B5：完成函式後執行，保存結果供後面的本機檢查使用。
    raise NotImplementedError("請將本段改成呼叫並保存 beginner_result。")

In [ ]:
def check_beginner_output(result, max_chars=250):
    # 這些是可確定的格式檢查；比喻、事實與是否給出理解題答案需人工閱讀。
    return {
        "字元數符合": len(result.replace("\n", "")) <= max_chars,
        "保留專有名詞": "Embedding" in result,
        "必要標題": "## 白話說明" in result and "## 想一想" in result,
    }

if RUN_PRACTICE_B and "beginner_result" in globals():
    print(check_beginner_output(beginner_result))

### B觀察紀錄
| 原文必要事實 | 實際輸出 | 字元／標題檢查 | 人工覆核 | 修改理由 |
|---|---|---|---|---|
|  |  |  |  |  |
- [ ] 保留Embedding且解釋符合原文。
- [ ] 250字元內，有白話說明與想一想。
- [ ] 有一題理解題、不附答案；沒有新增無根據的技術細節。

## 📖 練習C：依據課程規定問答（與B擇一）
**情境**：回答questions裡的三題，policy是唯一資料來源，不可推測未提供的規定。
**已提供**：policy、questions、API helper及固定資料不足檢查；只修改函式與問題迴圈。
| 步驟 | TODO | 實際要完成 |
|---|---|---|
| C1 | instructions | 課程助教身分、只依policy、可回答的方式、代入missing_reply |
| C2 | input／呼叫／回傳 | 分隔policy與question，呼叫ask_ai，return回答str |
| C3 | 執行區 | 建立policy_answers，依questions順序呼叫、保存每題回答並印出 |
**執行**：讀下格完整示範（可不付費）→完成C1/C2/C3→執行定義→開RUN_PRACTICE_C→下一格檢查。
示範2次、練習3次請求；完成C觀察表後跳同儕覆核，不必執行B。
policy是練習資料，不代表其他周的正式專題規定。

In [ ]:
def answer_from_context(
    question,
    context,
    missing_reply="資料不足，請詢問授課教師。",
):
    # 這是 RAG 問答的雛形：答案只能來自 context，找不到就回固定文字。
    instructions = f"""
# Identity
你是課程資訊問答助理。
# Instructions
- 只能根據 `<course_info>` 回答，不可用常識補充規定。
- 無法回答時固定回覆「{missing_reply}」
- 可以回答時先給直接答案，再附簡短依據。
"""
    # context 與 question 分開包標籤，讓模型知道哪段是資料、哪段是問題。
    data = f"<course_info>{context}</course_info><question>{question}</question>"
    return ask_ai(data, instructions)


course_rules = "作業二截止時間為10月15日23:59，繳交平台為Moodle。"

RUN_CONTEXT_DEMO = False
if RUN_CONTEXT_DEMO:
    print(answer_from_context("作業二要交到哪裡？", course_rules))
    print(answer_from_context("作業二占學期成績幾分？", course_rules))

### C的輸入、回傳與三題成果
| 變數 | 型別／用途 |
|---|---|
| question | str，本次問題；由迴圈逐題傳入 |
| policy | 已提供的str，唯一依據；不得自行補成績比例 |
| missing_reply | str，找不到答案時的固定文字 |
| 函式回傳 | 一題回答str，不是list或print的None |
| policy_answers | list[str]，按questions順序保存三題完整回答 |
第一題應指出個人製作、第二題每人4分鐘、第三題固定資料不足。
前兩題措辭可不同；第三題須與missing_reply相符，不加前言。程式只精確檢查第三題，前兩題人工核對。
執行區只印答案不足以完成C3：須把每題回傳文字存入policy_answers，讓下格能檢查。

In [ ]:
# 練習 C：policy 是唯一可用資料來源，回答不可使用常識補充。
policy = """
期中專題採個人製作，於第8週課堂展示。作品需包含簡單操作介面、
至少一次 API 呼叫與錯誤處理。展示時間每人4分鐘。
"""
questions = ["期中專題可以兩人一組嗎？", "每人展示多久？", "期中專題占總成績幾分？"]


def answer_policy_question(
    question,
    missing_reply="資料不足，請詢問授課教師。",
):
    # TODO C1：建立 instructions，只能依 policy 回答，並代入 missing_reply
    # 提示：明確要求「找不到答案時固定回覆 missing_reply」。
    # TODO C2：組合 policy 與 question，呼叫 ask_ai，再 return 回答文字
    # 提示：用 <policy> 與 <question> 分隔資料和問題。
    raise NotImplementedError("請完成資料問答。")


RUN_PRACTICE_C = False
if RUN_PRACTICE_C:
    # TODO C3：逐題呼叫並將回答依序保存到 policy_answers，供本機檢查使用。
    raise NotImplementedError("請完成問題迴圈。")

In [ ]:
if RUN_PRACTICE_C and "policy_answers" in globals():
    # 資料不足可做精確檢查；有答案的事實正確性仍需人工核對。
    assert len(policy_answers) == 3
    assert policy_answers[2].strip() == "資料不足，請詢問授課教師。"
    print("資料不足固定回覆檢查通過；請人工確認個人製作與每人4分鐘。")

### C觀察紀錄
| 問題 | 預期事實／行為 | 實際回答 | 通過／修正 |
|---|---|---|---|
| 可兩人一組嗎？ | 個人製作 |  |  |
| 展示多久？ | 每人4分鐘 |  |  |
| 占總成績幾分？ | 固定資料不足 |  |  |
- [ ] 有答案時忠於policy，措辭可不同。
- [ ] 沒答案時不猜測，固定回覆通過程式檢查。

### 同儕檢查（135–150分）
交換規則與測試紀錄：對方能否不用猜測就判斷衝突標籤？範例是否誤把測試答案洩漏？
指出一個失敗或仍未驗證的條件，完成一次修改並寫下原因。

## 🌱 延伸選做：v3增加衝突範例
完成A及B/C一題後才做；不影響課堂完成。固定規則與測試題維持原樣。
**TODO V3**：SUPPORT_EXAMPLES_V3先保留全部v2配對，在末尾增加一筆新衝突配對。
新配對須符合帳務→功能→操作→其他優先序，不得抄固定測試題；不修改support_v3或測試器。
執行定義→RUN_SUPPORT_V3改True→同五題測試，共5次API請求→填選做表。

In [ ]:
# TODO V3：保留v2範例並加入一筆衝突範例。
SUPPORT_EXAMPLES_V3 = []


def support_v3(text):
    return classify_support_message(text, SUPPORT_EXAMPLES_V3)

RUN_SUPPORT_V3 = False
if RUN_SUPPORT_V3:
    if SUPPORT_EXAMPLES_V3[:len(SUPPORT_EXAMPLES_V2)] != SUPPORT_EXAMPLES_V2 or len(SUPPORT_EXAMPLES_V3) <= len(SUPPORT_EXAMPLES_V2):
        raise ValueError("請保留v2範例並至少增加一筆衝突範例。")
    version_results["v3"] = run_classification_tests(SUPPORT_TESTS, support_v3, "v3")

| 選做版本 | 修改 | 通過 | 保留／退回與原因 |
|---|---|---|---|
| v3 | 在v2增加衝突範例 | /5 |  |
v3也可能與v2同分；記錄觀察，不宣稱必然改善。

## ✅ 課堂完成檢核
- [ ] 理解工作身分與API role、五元素、資料分隔與任務拆解。
- [ ] 引導練習：9則訊息與最後user檢查通過。
- [ ] 練習A：客服分類器合法標籤、每類一例、v1/v2五題比較與紀錄。
- [ ] 練習B或C：完成所選函式、程式檢查與人工覆核。
- [ ] 能說明相同結果也可能合理，並指出範例提供了什麼資訊。
v3、未選題與額外挑戰都是選做，不列入課堂必做檢核。

## 📝 課後繳交與第五周銜接
必交：訊息組裝結果、客服v1/v2規則／範例、五題比較表、B或C其中一題的輸出與檢查，
模型名稱及150字修訂反思。v3與未選題可另附選做成果，不要求全部完成。
不提交金鑰、個資或其他敏感資料。
第五周把人工user/assistant範例延伸成真實對話history；第六周再教Structured Outputs。

## 📚 延伸閱讀

- OpenAI Prompt engineering：https://developers.openai.com/api/docs/guides/prompt-engineering
- OpenAI Text generation：https://developers.openai.com/api/docs/guides/text

下週將進入：**多輪對話、conversation state 與 Streaming 輸出**。
